In [ ]:
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from agents import Agent, Runner, trace

In [ ]:
params = {"command": "uv", "args": ["run", "rx_logic.py"]}

try:
    async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
        mcp_tools = await server.list_tools()

    if not mcp_tools:
        raise RuntimeError("MCP server started but returned no tools — check rx_logic.py for registration errors.")

    print(f"✓ {len(mcp_tools)} tools loaded: {[t.name for t in mcp_tools]}")
except FileNotFoundError:
    print("✗ Could not start MCP server — is 'uv' installed and rx_logic.py in the current directory?")
    raise
except Exception as e:
    print(f"✗ MCP server error: {type(e).__name__}: {e}")
    raise

In [ ]:
INSTRUCTIONS = """
You are a clinical pharmacy validation assistant. You have access to a set of MCP tools
backed by the RxNorm API and the FDA openFDA API. Your role is to assist pharmacists
and clinicians in reviewing prescriptions for safety issues — not to make final clinical
decisions.

## Capabilities

You can validate a prescription against all of the following dimensions:
- Drug name normalisation (RxNorm)
- Allergy and cross-reactivity checking
- Active FDA recalls
- Pregnancy safety (label-based)
- Renal dose adjustment (CrCl-guided)
- Paediatric appropriateness
- Geriatric considerations (simplified Beers-style)
- Drug–drug interactions (pairwise label search)
- Multi-drug interaction scan across a full medication list
- Therapeutic and exact-duplicate detection (ATC-based and generic-based)
- Daily dose calculation (standard oral frequency abbreviations)
- Controlled-substance DEA scheduling
- Full FDA label retrieval (contraindications, warnings, dosing, interactions, etc.)

## Mandatory workflow

Follow these steps in order for every prescription validation request:

1. **Normalise each drug name** using `normalize_drug_name`.
   - Capture `rxcui`, `generic_name`, `ingredients`, and `atc_classes` from the result.
   - Pass the returned dict as `normalized` to `check_drug_allergy` and as the
     `normalized` field inside each medication dict for `check_therapeutic_duplication`.
   - If normalisation fails (`success: false`), note it but proceed using the raw name.

2. **Run mandatory safety gates** (do not skip these):
   - `check_drug_allergy` — pass the patient's full allergy list.
   - `check_drug_recall` — pass lot number when available.

3. **Run patient-specific checks** (only when the relevant data is present):
   - `check_pregnancy_safety` — any pregnant patient; include `trimester` if known.
   - `check_renal_dosing` — when CrCl is documented or suspected to be impaired.
   - `check_pediatric_dosing` — patients under 18; include weight in kg when known.
   - `check_geriatric_considerations` — patients 65 and older.

4. **Check interactions and duplicates** for multi-drug regimens:
   - `check_drug_interaction(drug1, drug2)` for targeted pairwise checking.
   - `check_multi_drug_interactions(drugs)` to scan all pairs in a list.
   - `check_duplicate_therapy(medications)` and/or `check_therapeutic_duplication(medications)`
     to catch redundant therapy.

5. **Verify contraindications** for relevant patient conditions:
   - `check_contraindication(drug_name, patient_condition)` for each known condition.

6. **Calculate daily dose** when the dose and frequency are provided:
   - `calculate_daily_dose(dose_per_administration, frequency)`.

7. **Retrieve full label** when you need indications, full warnings, or storage details:
   - `get_drug_label_info(drug_name)`.

## Interpreting results — critical rules

- `has_interaction: null` → Label data was unavailable. **Do not say "no interaction found."**
  Report it as unverified and recommend consulting a drug-interaction database.
- `is_contraindicated: null` → Condition was not found in the label.
  **Do not say the drug is safe for that condition.** Report as unverified.
- `has_allergy: false` → No substring match in this heuristic.
  **Do not say the patient is not allergic.** This check is not exhaustive.
- `has_recall: null` → The recall API call failed; status is unknown, not clear.
- Any empty `{}` label return → The FDA label was not found; do not infer safety.

## Communicating findings

Structure your response as a validation report with these sections:
1. **Drug identity** — normalised name, RxCUI, generic name, ATC class(es).
2. **Safety alerts** — allergy hits, active recalls, contraindications. Flag anything with
   `CRITICAL` or `DO NOT DISPENSE` prominently.
3. **Patient-specific concerns** — pregnancy, renal, paediatric, geriatric (as applicable).
4. **Interactions** — list any pairs flagged; include severity and source.
5. **Duplicates** — exact or therapeutic class.
6. **Dosing** — calculated daily dose; flag unusual frequencies.
7. **Controlled substance status** — DEA schedule when applicable.
8. **Data gaps** — explicitly list every check that returned null or no label, and recommend
   consulting Lexicomp, Micromedex, or a licensed pharmacist for those gaps.

## Scope and disclaimer

These tools use public APIs and heuristic label-text parsing. They are **not** a replacement
for a licensed pharmacist, clinical decision-support system (Lexicomp, Micromedex,
DrugBank, FDB, Medi-Span), or institutional formulary review. Always surface the
`recommendation` strings returned by each tool — they are written with cautious language.
Tell the user to confirm findings with a qualified clinician before any dispensing decision.
"""


In [ ]:
REQUEST = """
Please perform a full prescription validation for the following patient and medication order.

## Patient
- Age: 72 years old
- Weight: 68 kg
- Sex: Female
- Pregnant: No
- Renal function: CrCl 22 mL/min (severe impairment)
- Known allergies: penicillin, sulfa drugs
- Active conditions: type 2 diabetes, hypertension, chronic kidney disease stage 4

## Prescription
- Drug: Metformin
- Strength / dose: 1000 mg per dose
- Route: Oral
- Frequency: BID (twice daily)
- Duration: 90 days
- Indication: glycaemic control

## Concurrent medications already prescribed
- Lisinopril 10 mg once daily
- Amlodipine 5 mg once daily
- Atorvastatin 40 mg once daily at bedtime

Please validate all of the following and provide a structured report:
1. Confirm the drug identity and generic name via RxNorm.
2. Check for allergy or cross-reactivity against her documented allergies.
3. Check for any active FDA recalls.
4. Assess whether Metformin is appropriate given CrCl 22 mL/min, and recommend dose adjustment or alternative if needed.
5. Check for relevant contraindications given her CKD stage 4 and diabetes.
6. Scan all four medications for drug–drug interactions.
7. Check for therapeutic duplication across the full regimen.
8. Calculate the total daily dose of Metformin as prescribed.
9. Note any geriatric considerations given her age.
10. Report any data gaps where findings could not be verified from available label data.
"""


In [ ]:
params = {"command": "uv", "args": ["run", "rx_logic.py"]}

try:
    async with MCPServerStdio(params=params, client_session_timeout_seconds=120) as rx_logic_server:
        agent = Agent(
            name="RxValidator",
            model="gpt-4.1-mini",
            instructions=INSTRUCTIONS,
            mcp_servers=[rx_logic_server],
        )
        with trace("Rx_validation"):
            result = await Runner.run(agent, REQUEST, max_turns=30)

    if result.final_output:
        display(Markdown(result.final_output))
    else:
        print("⚠ Agent returned no output. Check the trace for tool call failures.")

except TimeoutError:
    print("✗ MCP session timed out — increase client_session_timeout_seconds or simplify the request.")
    raise
except Exception as e:
    print(f"✗ Validation failed: {type(e).__name__}: {e}")
    raise